# Assemble submission
Reuse this notebook for each submission. Set NAME and MODEL_NAMES in the first code cell; the JSON card records the selected artifacts.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
from src.pipeline.random_seed import set_seed

SEED = set_seed()
N = 25_889
NAME = "kaggle_t12345_current_v5"  # Give each new submission artifact a new name.
MODEL_NAMES = {
    1: "t1_lgbm_combined_pruned_cv6_full_v1",
    2: "t2_cat_original_no_noise_v2",
    3: "t3_cat_combined_v1",
    4: "t4_lgbm_31_tuned_features_r2",
    5: "t5_r2_ocsvm_nu10_engineered_v2",
}
EXPECTED_METRICS = {1:"macro_F2_outer_cv6",2:"macro_F1",3:"MAE_thb",4:"macro_F1",5:"top5_rank_stability_jaccard"}
OUTDATED_MODEL_NAMES = {2: ['t2_cat_combined_no_noise_v1', 't2_cat_combined_no_tiers_v1', 't2_cat_no_engagement_v1', 't2_cat_no_loyalty_v1', 't2_cat_no_tiers_v1', 't2_cat_original_no_noise_v1', 't2_dummy_original_v1', 't2_hgb_combined_no_noise_tuned_v1', 't2_hgb_combined_no_noise_v1', 't2_hgb_combined_no_tiers_v1', 't2_hgb_combined_v1', 't2_hgb_no_engagement_v1', 't2_hgb_no_loyalty_v1', 't2_hgb_no_noise_tuned_v1', 't2_hgb_no_noise_v1', 't2_hgb_no_tiers_v1', 't2_hgb_original_v1', 't2_lgb_combined_no_noise_v1', 't2_lgb_original_no_noise_v1', 't2_logreg_combined_v1', 't2_logreg_original_v1', 't2_rf_combined_v1', 't2_rf_original_v1', 't2_xgb_combined_no_noise_v1', 't2_xgb_original_no_noise_v1'], 4: ['t4_dummy_pixels_r1', 't4_dummy_tuned_pixels_r1', 't4_extratrees_features_r1', 't4_extratrees_pixels_r1', 't4_extratrees_tuned_features_r1', 't4_extratrees_tuned_pixels_r1', 't4_hgb_features_r1', 't4_hgb_pixels_r1', 't4_hgb_tuned_features_r1', 't4_hgb_tuned_pixels_r1', 't4_lgbm_15_features_r1', 't4_lgbm_15_pixels_r1', 't4_lgbm_15_tuned_features_r1', 't4_lgbm_15_tuned_pixels_r1', 't4_lgbm_31_features_r1', 't4_lgbm_31_pixels_r1', 't4_lgbm_31_tuned_features_r1', 't4_lgbm_31_tuned_pixels_r1', 't4_lgbm_tuned_features_r1', 't4_lgbm_tuned_pixels_r1', 't4_logreg_features_r1', 't4_logreg_pixels_r1', 't4_logreg_tuned_features_r1', 't4_logreg_tuned_pixels_r1', 't4_svm_features_r1', 't4_svm_pixels_r1', 't4_svm_tuned_features_r1', 't4_svm_tuned_pixels_r1'], 5: ['t5_r1_gmm_engineered_v1', 't5_r1_gmm_k8_engineered_v1', 't5_r1_gmm_k8_original_v1', 't5_r1_gmm_original_v1', 't5_r1_iforest_engineered_v1', 't5_r1_iforest_ms512_engineered_v1', 't5_r1_iforest_ms512_original_v1', 't5_r1_iforest_original_v1', 't5_r1_lof_engineered_v1', 't5_r1_lof_original_v1', 't5_r1_ocsvm_engineered_v1', 't5_r1_ocsvm_nu10_engineered_v1', 't5_r1_ocsvm_nu10_original_v1', 't5_r1_ocsvm_original_v1', 't5_r1_pca_engineered_v1', 't5_r1_pca_original_v1', 't5_r1_tail_engineered_v1', 't5_r1_tail_original_v1']}
assert {task:len(names) for task,names in OUTDATED_MODEL_NAMES.items()} == {2:25,4:28,5:18}
for task, old_names in OUTDATED_MODEL_NAMES.items():
    pattern = "*_r1.json" if task == 4 else "*v1.json"
    current_inventory = {p.stem for p in (SRC / f"task{task}" / "save").glob(pattern)
                         if not p.stem.endswith("-evaluation")}
    assert current_inventory == set(old_names), f"Task {task} outdated inventory changed"
    assert MODEL_NAMES[task] not in old_names, f"Task {task} selected an outdated model"
print("Selected models:")
display(pd.Series(MODEL_NAMES, name="model_name").rename_axis("task").to_frame())

def load_task(task):
    name = MODEL_NAMES[task]
    save_dir = SRC / f"task{task}" / "save"
    model_card = json.loads((save_dir / f"{name}.json").read_text(encoding="utf-8"))
    evaluation = json.loads((save_dir / f"{name}-evaluation.json").read_text(encoding="utf-8"))
    assert model_card["name"] == evaluation["name"] == name
    data_card = json.loads((SRC / model_card["data_card_path"]).read_text(encoding="utf-8"))
    assert data_card["name"] == model_card["data_name"]
    test_path = SRC / data_card["splits"]["test"]["path"]
    test = pd.read_parquet(test_path) if data_card["format"] == "parquet" else pd.read_csv(test_path)
    id_col = "id" if task == 4 else "player_id"
    raw_ids = pd.read_csv(SRC / f"task{task}" / "data" / "raw" / "test.csv", usecols=[id_col])[id_col]
    assert len(test) == N and test[id_col].is_unique and test[id_col].notna().all()
    assert test[id_col].reset_index(drop=True).equals(raw_ids.reset_index(drop=True))
    assert evaluation["metric"] == EXPECTED_METRICS[task]
    if task in OUTDATED_MODEL_NAMES:
        assert name not in OUTDATED_MODEL_NAMES[task]
        assert model_card["data_name"].endswith("v2")
        assert "outdate" not in data_card["card"]["source"].lower()
        assert "test.csv" in data_card["card"]["source"]
    model = joblib.load(SRC / model_card["model_path"])
    source = {"model_name": name, "model_card_path": f"task{task}/save/{name}.json",
              "model_path": model_card["model_path"], "data_name": model_card["data_name"],
              "data_card_path": model_card["data_card_path"], "test_path": data_card["splits"]["test"]["path"],
              "evaluation_metric": evaluation["metric"], "evaluation_score": evaluation["score"],
              "test_rows": len(test), "test_id_first": str(test[id_col].iloc[0]),
              "test_id_last": str(test[id_col].iloc[-1])}
    return model, test, source


Selected models:


,model_name
task,
1,t1_lgbm_combined_pruned_cv6_full_v1
2,t2_cat_original_no_noise_v2
3,t3_cat_combined_v1
4,t4_lgbm_31_tuned_features_r2
5,t5_r2_ocsvm_nu10_engineered_v2


## Predict each task

In [2]:
model1, test1, source1 = load_task(1)
pred1 = np.asarray(model1.predict(test1.drop(columns="player_id"))).reshape(-1)
assert len(pred1) == N and set(np.unique(pred1)) <= {0, 1}
print("Task 1:", pd.Series(pred1).value_counts().sort_index().to_dict())

Task 1: {0: 15944, 1: 9945}


In [3]:
model2, test2, source2 = load_task(2)
features2 = test2.drop(columns="player_id").copy()
categorical_columns = model2.named_steps["preprocess"].transformers_[0][2]
for column in categorical_columns:
    features2[column] = features2[column].astype(object).where(features2[column].notna(), None)
pred2 = np.asarray(model2.predict(features2)).reshape(-1)
assert len(pred2) == N and set(np.unique(pred2)) <= {0, 1, 2, 3}
print("Task 2:", pd.Series(pred2).value_counts().sort_index().to_dict())

Task 2: {0: 11308, 1: 6381, 2: 4692, 3: 3508}


In [4]:
model3, test3, source3 = load_task(3)
pred3_raw = np.asarray(model3.predict(test3.drop(columns="player_id")), dtype=float).reshape(-1)
assert len(pred3_raw) == N and np.isfinite(pred3_raw).all()
pred3 = np.maximum(pred3_raw, 0.0)  # The existing v2 validated this clipping on held-out data.
print("Task 3: negative raw predictions", int((pred3_raw < 0).sum()),
      "clipped range", (round(float(pred3.min()), 2), round(float(pred3.max()), 2)))

Task 3: negative raw predictions 3125 clipped range (0.0, 242435.75)


In [5]:
model4, test4, source4 = load_task(4)
features4 = test4.drop(columns=["id", "class", "image_sha256", "width", "height"])
assert features4.columns.tolist() == model4.feature_name_
pred4 = np.asarray(model4.predict(features4.astype("float32"))).reshape(-1)
assert len(pred4) == N and set(pred4) <= {"7-11k", "fiveN", "freefried", "pube", "roblock"}
# The competition Task 4 datacard defines the numeric submission classes.
TASK4_CLASS_TO_INT = {"7-11k": 0, "fiveN": 1, "freefried": 2, "pube": 3, "roblock": 4}
current_train = pd.read_csv(SRC / "task4/data/raw/train.csv")
old_train = pd.read_csv(SRC / "task4/data/raw/train-outdate-12-11-2026.csv")
matched = current_train.merge(old_train, left_on=current_train.file_name.str.removesuffix(".jpg"), right_on="id", how="left")
assert len(matched) == len(current_train) and matched["class"].notna().all()
assert matched["class"].map(TASK4_CLASS_TO_INT).equals(matched["label"])
pred4 = pd.Series(pred4).map(TASK4_CLASS_TO_INT).to_numpy(dtype=int)
assert set(np.unique(pred4)) <= {0, 1, 2, 3, 4}
print("Task 4 numeric classes:", pd.Series(pred4).value_counts().sort_index().to_dict())

Task 4 numeric classes: {0: 8384, 1: 3686, 2: 4169, 3: 7783, 4: 1867}


In [6]:
model5, test5, source5 = load_task(5)
pred5 = np.asarray(model5.predict(test5.drop(columns="player_id"))).reshape(-1)
assert len(pred5) == N and set(np.unique(pred5)) <= {0, 1}
print("Task 5 model:", pd.Series(pred5).value_counts().sort_index().to_dict())
print("Task 5 evaluation is a rank-stability proxy, not F3:", source5["evaluation_score"])

Task 5 model: {0: 24524, 1: 1365}
Task 5 evaluation is a rank-stability proxy, not F3: 1.0


## Assemble, compare with v2, and save

In [7]:
sample = pd.read_csv(SRC / "submission" / "sample_submission.csv")
assert sample.shape == (N, 6) and sample.columns.tolist() == ["id", "task1", "task2", "task3", "task4", "task5"]
ans_ids = sample["id"].tolist()
assert ans_ids == [f"ANS{i:05d}" for i in range(1, N + 1)]
assert test4.id.tolist() == ans_ids
submission = pd.DataFrame({"id": ans_ids, "task1": pred1.astype(int), "task2": pred2.astype(int),
                           "task3": pred3, "task4": pred4, "task5": pred5.astype(int)})
expected_cols = ["id", "task1", "task2", "task3", "task4", "task5"]
assert submission.shape == (N, 6) and submission.columns.tolist() == expected_cols
assert submission.id.is_unique and not submission.isna().any().any() and (submission.task3 >= 0).all()
prior = pd.read_csv(SRC / "submission/kaggle_t12345_best_v4.csv")
assert prior.id.tolist() == ans_ids
assert np.array_equal(prior.task1.to_numpy(), submission.task1.to_numpy())
assert np.allclose(prior.task3.to_numpy(), submission.task3.to_numpy(), rtol=0, atol=1e-8)
for task, old_names in OUTDATED_MODEL_NAMES.items():
    assert MODEL_NAMES[task] not in old_names
print("Outdated model selection check passed")
display(submission.head())

Outdated model selection check passed


,id,task1,task2,task3,task4,task5
0,ANS00001,1,3,4.412224e-07,2,1
1,ANS00002,1,2,2.638772e+03,3,0
2,ANS00003,1,2,3.886101e-07,3,0
3,ANS00004,1,1,0.000000e+00,0,0
4,ANS00005,0,0,9.299521e+01,3,0


In [8]:
output_dir = SRC / "submission"
csv_path, card_path = output_dir / f"{NAME}.csv", output_dir / f"{NAME}.json"
assert csv_path.exists() == card_path.exists(), "Incomplete submission artifact"
sources = {f"task{task}": source for task, source in enumerate((source1, source2, source3, source4, source5), start=1)}
for task, old_names in OUTDATED_MODEL_NAMES.items():
    assert sources[f"task{task}"]["model_name"] not in old_names
    assert sources[f"task{task}"]["data_name"].endswith("v2")
if not csv_path.exists():
    submission.to_csv(csv_path, index=False)
    card = {
        "name": NAME, "created_at": datetime.now(timezone.utc).isoformat(),
        "card": "Refreshed Task 2/4/5 model submission; Task 5 selected by unlabeled proxy",
        "submission_path": f"submission/{NAME}.csv", "format": "comma-separated CSV",
        "rows": len(submission), "columns": expected_cols,
        "sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(), "sources": sources,
        "outdated_model_check": {"tasks": [2, 4, 5], "fixed_counts": {str(t): len(v) for t, v in OUTDATED_MODEL_NAMES.items()}, "selected_models_excluded": True},
        "task2_input_adjustment": "Restore categorical nulls to None for saved imputer",
        "task3_postprocessing": "Clip negative predictions to zero as previously validated",
        "task4_label_mapping": TASK4_CLASS_TO_INT,
        "id_mapping": "Task 4 current ANS IDs checked directly; Tasks 1/2/3/5 use Test row position",
        "task5_metric_note": "Rank stability Jaccard is a proxy, not macro F3",
    }
    card_path.write_text(json.dumps(card, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
else:
    card = json.loads(card_path.read_text(encoding="utf-8"))
    assert card["name"] == NAME and card["sources"] == sources
    assert card["sha256"] == hashlib.sha256(csv_path.read_bytes()).hexdigest()
saved = pd.read_csv(csv_path)
assert saved.shape == submission.shape and saved.columns.tolist() == expected_cols
assert saved.id.tolist() == ans_ids
for column in ("task1", "task2", "task4", "task5"):
    assert np.array_equal(saved[column].to_numpy(), submission[column].to_numpy())
assert np.allclose(saved.task3.to_numpy(), submission.task3.to_numpy(), rtol=0, atol=1e-8)
print("Verified", csv_path, "SHA-256", card["sha256"])


Verified C:\Users\BTCOM\Desktop\ml-final\src\submission\kaggle_t12345_current_v5.csv SHA-256 c9fb803446cb5ef66fe05f5c8ac319afb351ba7cecac64e8f3b52b7afaa9aa47
